In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Install Gradio
!pip install gradio


In [ ]:
import torch

In [ ]:

# Load the full model
model = torch.load('/content/drive/MyDrive/final_pruned_model.pth', map_location='cpu')

# Let's check what we got
print("Type:", type(model))
print("\n")

# If it's a full model (not just state_dict)
if hasattr(model, 'eval'):
    model.eval()
    print("✅ It's a full model!")
    print(model)  # This will show the full architecture
else:
    print("⚠️ It's a state_dict (just weights)")
    print(model.keys())


Type: <class 'collections.OrderedDict'>


⚠️ It's a state_dict (just weights)
odict_keys(['efficientnet.features.0.0.weight', 'efficientnet.features.0.1.weight', 'efficientnet.features.0.1.bias', 'efficientnet.features.0.1.running_mean', 'efficientnet.features.0.1.running_var', 'efficientnet.features.0.1.num_batches_tracked', 'efficientnet.features.1.0.block.0.0.weight', 'efficientnet.features.1.0.block.0.1.weight', 'efficientnet.features.1.0.block.0.1.bias', 'efficientnet.features.1.0.block.0.1.running_mean', 'efficientnet.features.1.0.block.0.1.running_var', 'efficientnet.features.1.0.block.0.1.num_batches_tracked', 'efficientnet.features.1.0.block.1.fc1.weight', 'efficientnet.features.1.0.block.1.fc1.bias', 'efficientnet.features.1.0.block.1.fc2.weight', 'efficientnet.features.1.0.block.1.fc2.bias', 'efficientnet.features.1.0.block.2.0.weight', 'efficientnet.features.1.0.block.2.1.weight', 'efficientnet.features.1.0.block.2.1.bias', 'efficientnet.features.1.0.block.2.1.running_mean'

In [ ]:
state_dict = torch.load('/content/drive/MyDrive/final_pruned_model.pth', map_location='cpu')

# Check number of output classes
print("Number of classes:", state_dict['efficientnet.classifier.1.weight'].shape[0])
print("Classifier input features:", state_dict['efficientnet.classifier.1.weight'].shape[1])

# Check first conv layer to help identify EfficientNet version
print("First conv shape:", state_dict['efficientnet.features.0.0.weight'].shape)

Number of classes: 4
Classifier input features: 1280
First conv shape: torch.Size([32, 3, 3, 3])


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class AlzheimerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.efficientnet = models.efficientnet_b0(weights=None)
        self.efficientnet.classifier[1] = nn.Linear(1280, 4)

    def forward(self, x):
        return self.efficientnet(x)

model = AlzheimerModel()
state_dict = torch.load('/content/drive/MyDrive/final_pruned_model.pth', map_location='cpu')

try:
    model.load_state_dict(state_dict)
    model.eval()
    print("✅ Model loaded successfully! We're good to go!")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Model loaded successfully! We're good to go!


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import gradio as gr

# ============ 1. MODEL ============
class AlzheimerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.efficientnet = models.efficientnet_b0(weights=None)
        self.efficientnet.classifier[1] = nn.Linear(1280, 4)

    def forward(self, x):
        return self.efficientnet(x)

model = AlzheimerModel()
state_dict = torch.load('/content/drive/MyDrive/final_pruned_model.pth', map_location='cpu')
model.load_state_dict(state_dict)
model.eval()
print("✅ Model loaded!")

# ============ 2. LABELS ============
class_names = ['Mild Demented', 'Moderate Demented', 'Non Demented', 'Very Mild Demented']

# ============ 3. PREPROCESSING ============
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ============ 4. PREDICT ============
def predict(image):
    img = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)[0]
    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

# ============ 5. GRADIO ============
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=4),
    title="🧠 Alzheimer's Disease Detection",
    description="""
    Upload a brain MRI scan to classify the stage of Alzheimer's Disease.

    **Classes:**
    - 🟢 Non Demented
    - 🟡 Very Mild Demented
    - 🟠 Mild Demented
    - 🔴 Moderate Demented

    *Powered by EfficientNet-B0 (Pruned - Lottery Ticket Method)*
    """,
    theme="default"
)

demo.launch(share=True)

✅ Model loaded!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cbe74f4fc49066c8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd

# Use the same labels from your dataset
labels = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
codes = pd.Categorical(labels).codes
print("Label mapping:")
for label, code in zip(labels, codes):
    print(f"  {code} → {label}")